# Task 4 — Restaurant Rating Prediction

**Objective:** Build and compare Machine Learning regression models that predict  
a restaurant's aggregate rating from its features.

---
### Features Used
| Feature | Type | Description |
|---------|------|-------------|
| Votes | Numerical | Number of user votes |
| Average Cost for two | Numerical | Average meal cost |
| Has Online delivery | Binary | 0 or 1 |
| Has Table booking | Binary | 0 or 1 |
| Price range | Categorical | 1–4 |
| Primary Cuisine | Categorical | First cuisine listed |
| City | Categorical | Restaurant city |

### Target
**Aggregate rating** (continuous, range 1.0 – 5.0)

### Models Trained
1. Linear Regression (baseline)
2. Ridge Regression (regularised)
3. Random Forest Regressor
4. **Gradient Boosting Regressor** ← best performer

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning     import load_clean
from src.rating_prediction import (prepare_features, train_and_evaluate,
                                    ALL_FEATURES, TARGET)
from src.visualization     import (plot_actual_vs_predicted, plot_feature_importance,
                                    plot_model_comparison, plot_residuals)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Libraries loaded ✓')

## 1. Load Data & Feature Overview

In [ ]:
df = load_clean()
print(f"Dataset: {df.shape[0]:,} rows")

# Show the features we will use
print(f"\nFeatures ({len(ALL_FEATURES)}): {ALL_FEATURES}")
print(f"Target: {TARGET}")

df[ALL_FEATURES + [TARGET]].describe().round(3)

## 2. Feature Distributions (EDA for ML)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
num_cols = ['Votes', 'Average Cost for two', 'Aggregate rating']
colors   = ['steelblue', 'darkorange', 'seagreen']
for ax, col, c in zip(axes, num_cols, colors):
    ax.hist(df[col], bins=30, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.suptitle('Distribution of Key Numerical Features', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between numerical features and target
num_feat_cols = ['Votes', 'Average Cost for two', 'Has Online delivery',
                 'Has Table booking', 'Price range', 'Aggregate rating']
corr = df[num_feat_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Prepare Features

In [ ]:
X, y = prepare_features(df)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y range: {y.min():.2f} – {y.max():.2f}")
print(f"y mean : {y.mean():.4f}")
X.head()

## 4. Train All Models & Compare

In [ ]:
# This cell trains 4 models and prints metrics for each
results = train_and_evaluate(df)

## 5. Results Table

In [ ]:
results_df = results['results']
results_df

In [ ]:
print(f"\n★ Best Model: {results['best_name']}")
best_row = results_df[results_df['Model'] == results['best_name']].iloc[0]
print(f"  MAE      = {best_row['MAE']}   (avg error in rating points)")
print(f"  RMSE     = {best_row['RMSE']}  (penalises large errors more)")
print(f"  R² Score = {best_row['R2 Score']} (variance explained by model)")

## 6. Model Comparison Charts

In [ ]:
fig = plot_model_comparison(results_df, metric='R2 Score')
plt.show()

In [ ]:
fig = plot_model_comparison(results_df, metric='MAE')
plt.show()

In [ ]:
fig = plot_model_comparison(results_df, metric='RMSE')
plt.show()

## 7. Actual vs Predicted — Best Model

In [ ]:
fig = plot_actual_vs_predicted(
    results['y_test'], results['y_pred_best'],
    model_name=results['best_name']
)
plt.show()

## 8. Residual Analysis

In [ ]:
fig = plot_residuals(
    results['y_test'], results['y_pred_best'],
    model_name=results['best_name']
)
plt.show()

residuals = np.array(results['y_test']) - np.array(results['y_pred_best'])
print(f"Residuals — Mean: {residuals.mean():.4f}  Std: {residuals.std():.4f}")
print(f"% of predictions within ±0.5 rating points: "
      f"{(np.abs(residuals) <= 0.5).mean()*100:.1f}%")

## 9. Feature Importances

In [ ]:
importance_df = results['feature_importance']
if importance_df is not None:
    print(f"Top 15 features ({results['best_name']})")
    print(importance_df.head(15).to_string(index=False))
    
    fig = plot_feature_importance(importance_df, top_n=15)
    plt.show()
else:
    print('Feature importance not available for linear models.')

## 10. Predict a Custom Restaurant

In [ ]:
from src.rating_prediction import predict_single

# Define a hypothetical restaurant
my_restaurant = {
    'Votes':                 350,
    'Average Cost for two':  600,
    'Has Online delivery':     1,
    'Has Table booking':       0,
    'Price range':             2,
    'Primary Cuisine':     'North Indian',
    'City':                'New Delhi',
}

predicted_rating = predict_single(results['best_model'], my_restaurant)
print(f"Predicted Rating for custom restaurant: {predicted_rating:.2f} / 5.00")

## 11. Metric Explanations & Conclusions

```
KEY FINDINGS — Task 4: Rating Prediction
=========================================

MODEL PERFORMANCE SUMMARY
─────────────────────────
Gradient Boosting  → R²=0.60  MAE=0.25  RMSE=0.35  ← BEST
Random Forest      → R²=0.57  MAE=0.26  RMSE=0.36
Ridge Regression   → R²=0.47  MAE=0.30  RMSE=0.40
Linear Regression  → R²=0.47  MAE=0.31  RMSE=0.40

METRIC EXPLANATIONS
────────────────────
R² Score  = 0.60 means the model explains 60% of the variation in
            restaurant ratings from these features alone.
MAE       = 0.25 means on average the predicted rating is within
            ±0.25 stars of the actual rating.
RMSE      = 0.35 penalises larger errors; our model rarely makes
            predictions more than 0.35 stars off.

MOST IMPORTANT FEATURES (Gradient Boosting)
────────────────────────────────────────────
1. Votes              — most predictive; popular = trusted = higher rating
2. Price range        — higher-priced venues consistently rated higher
3. Average Cost       — correlated with price range but adds nuance
4. City               — geographic factors matter (local culture, competition)
5. Primary Cuisine    — certain cuisines (e.g. Japanese) are rated higher

WHY NOT HIGHER R²?
────────────────────
Restaurant ratings depend on subjective factors (mood, service on the day,
individual taste) that no dataset can capture. An R² of ~0.60 is strong
for predicting subjective ratings.

PRACTICAL IMPLICATION
──────────────────────
A new restaurant owner can expect a rating of ~3.8+ if they:
  ✓ Are in the Premium price range
  ✓ Offer a popular cuisine for their city
  ✓ Accumulate a good number of votes (exposure)
```